# MILO — AI Nutrition Chef 🍳
### Proyecto final · IA Generativa: asistente experto con Gemini, RAG y agentes

<img src="assets/milo_small.png" width="190" align="right">

> **"Estos son mis tres documentos. Así los divido. Así los convierto en vectores. Aquí los guardo.
> Cuando pregunto algo, estos son los fragmentos que recupera. Y esta es la respuesta que genera Gemini
> utilizando esa información."**

Este notebook sigue exactamente esa frase: cada sección enseña una fase del RAG con datos reales, y al final
conecto todo en un agente de LangGraph con memoria.

## 1. Introducción

**MILO** es un pequeño chef digital especializado en cocina casera y nutrición. Le puedo decir qué tengo en la
nevera, mi objetivo de calorías o mis restricciones, y me recomienda recetas, calcula calorías y macronutrientes,
sustituye ingredientes, organiza un menú semanal y genera la lista de la compra.

**¿Qué problema resuelve?** Mucha gente no sabe qué cocinar con lo que tiene, cómo ajustar una receta a sus
objetivos o cómo organizar la semana. MILO combina conocimiento documental (RAG) con cálculos nutricionales
deterministas en Python.

| Pieza | Tecnología | Dónde está |
|---|---|---|
| Modelo de lenguaje | Google Gemini (`gemini-3.5-flash-lite`) | `src/milo/rag.py` → `create_model` |
| Embeddings | Gemini Embeddings (`gemini-embedding-001`, 3072 dimensiones) | `src/milo/rag.py` → `create_embeddings` |
| Base vectorial | ChromaDB persistente en `storage/chroma` | `src/milo/rag.py` → `open_index` |
| Agente | LangGraph (sobre LangChain) | `src/milo/agent.py` |
| Memoria | Checkpointer SQLite de LangGraph | `storage/memoria.sqlite` |
| Cálculos nutricionales | Python + 70 alimentos USDA | `src/milo/nutrition.py`, `src/milo/recipes.py` |
| System prompt | Archivo Markdown | `src/milo/prompts/system.md` |
| Interfaz web (bonus) | Streamlit | `app.py` |

Una idea clave del diseño: **el RAG busca conocimiento, Python hace las cuentas**. Gemini nunca calcula calorías.

## 2. Preparación del entorno

Instalo las dependencias con `python -m pip install -r requirements.txt` y guardo mi clave de Gemini en el archivo
`.env` (copia de `.env.example`). La clave nunca aparece en el código ni en este notebook.

In [1]:
import sys
import json
from collections import Counter
from pathlib import Path

sys.path.insert(0, "src")  # Por si el paquete milo no está instalado en el entorno.
import pandas as pd

from milo.config import ROOT, SOURCES_JSON, Settings
from milo.rag import (NutritionRAG, cited_numbers, create_embeddings, format_context, index_chunks,
                      load_documents, load_system_prompt, show_answer, rag_sources, location)
from milo.agent import MiloAgent, show_turn

settings = Settings()
settings.require_key()  # Falla con un mensaje claro si falta GOOGLE_API_KEY (no muestra la clave).
pd.set_option("display.max_colwidth", 90)
print("Clave de Gemini encontrada ✔")
print("Modelo de lenguaje:", settings.model, "| respaldo:", settings.fallback_model)
print("Modelo de embeddings:", settings.embedding_model, f"({settings.embedding_dimension} dimensiones)")
print("Carpeta de Chroma y memoria:", settings.storage_dir.relative_to(ROOT))

Clave de Gemini encontrada ✔
Modelo de lenguaje: gemini-3.5-flash-lite | respaldo: gemini-3.1-flash-lite
Modelo de embeddings: gemini-embedding-001 (3072 dimensiones)
Carpeta de Chroma y memoria: storage


## 3. "Estos son mis tres documentos"

He construido la base de conocimiento con tres documentos de la carpeta `data/documents/`:

1. **Guía oficial de AESAN (2022)** — *Recomendaciones dietéticas saludables y sostenibles*. PDF de 19 páginas del
   Ministerio de Consumo: raciones recomendadas de cada grupo de alimentos, frecuencias semanales, sal, azúcar, agua
   y actividad física.
2. **Recetario de MILO** — documento propio con 28 recetas: ingredientes en gramos (indicando si se pesan crudos,
   secos o cocinados), pasos, tiempo, dieta y alérgenos. Se genera a partir de `data/recipes.json`.
3. **Técnicas culinarias y sustituciones** — documento propio sobre cómo pesar, cocer, saltear y hornear, higiene,
   temperaturas seguras (FoodSafety.gov) y cómo sustituir ingredientes sin tratarlos como equivalentes exactos.

Cada documento está registrado en `data/sources.json` con su URL, versión y huella **SHA-256**: si alguien modifica
un archivo sin registrarlo, la ingesta se detiene.

In [2]:
fuentes = rag_sources()
pd.DataFrame([{
    "id": f.source_id, "título": f.title, "tipo": f.file_type, "archivo": f.path,
    "tamaño (KB)": round(f.size_bytes / 1024), "páginas omitidas": f.skip_pages or "-",
} for f in fuentes])

,id,título,tipo,archivo,tamaño (KB),páginas omitidas
0,AESAN_2022,Recomendaciones dietéticas saludables y sostenibles (AESAN),pdf,data/documents/aesan_recomendaciones_dieteticas_2022.pdf,8997,"[1, 2, 19]"
1,MILO_RECETAS,Mi recetario revisado para MILO,markdown,data/documents/recetario_milo.md,23,-
2,MILO_TECNICAS,Técnicas culinarias y sustituciones para MILO,markdown,data/documents/tecnicas_y_sustituciones.md,7,-


In [3]:
from pypdf import PdfReader

aesan = next(f for f in fuentes if f.source_id == "AESAN_2022")
pdf = PdfReader(ROOT / aesan.path)
print(f"El PDF de AESAN tiene {len(pdf.pages)} páginas. Texto extraído de la página 9:\n")
print(pdf.pages[8].extract_text()[:700])

El PDF de AESAN tiene 19 páginas. Texto extraído de la página 9:

9
HUEVOS CON MODERACIÓN, 
SEGÚN TUS NECESIDADES
Se recomienda consumir un máximo de 4 hue-
vos medianos a la semana.
Un huevo mediando M pesa entre 53-63 gramos. 
ADEMÁS…
Acompaña el consumo de huevos con alimentos 
saludables, como ensaladas, hortalizas, pescados y 
evita combinaciones con alimentos ricos en grasas 
saturadas, como algunas carnes procesadas y hari-
nas refi  nadas, como el pan blanco o los empanados.
Los huevos son un alimento asequible y con un 
impacto medioambiental relativamente bajo. 
Por razones de bienestar animal, si puedes, con-
sume huevos procedentes de gallinas criadas en 
libertad (camperas).
QUÉ DEBE ESTAR PRESENTE EN 
UNA DIETA SALUDABLE Y SOSTENIBLE
El embar


## 4. "Así los divido": preparación y chunking

Un documento entero es demasiado largo para compararlo con una pregunta concreta, así que lo divido en
**fragmentos (chunks)**:

- Limpio el texto extraído (guiones de final de línea, espacios repetidos, caracteres de control).
- Separo por títulos: `## ...` en Markdown y líneas en MAYÚSCULAS en el PDF. Así cada fragmento trata un tema.
- Si una sección supera **400 palabras**, la corto en ventanas con **50 palabras de solapamiento** para no perder
  frases en el límite.
- En el PDF uno las secciones muy cortas (un título suelto) con la siguiente.
- Omito la portada, el índice y la contraportada del PDF (páginas 1, 2 y 19): no aportan contenido y, en una primera
  versión, el índice aparecía como primer resultado de muchas búsquedas.
- Cada fragmento guarda su **procedencia**: documento, página (PDF) o sección (Markdown), versión y hash.

In [4]:
fragmentos = load_documents()
tabla = pd.DataFrame([{"documento": f.source_id, "palabras": len(f.text.split())} for f in fragmentos])
resumen = tabla.groupby("documento")["palabras"].agg(fragmentos="count", palabras_totales="sum", media="mean", máximo="max")
print("Total de fragmentos:", len(fragmentos))
resumen.round(0)

Total de fragmentos: 75


,fragmentos,palabras_totales,media,máximo
documento,,,,
AESAN_2022,35,4137,118.0,374
MILO_RECETAS,29,3725,128.0,172
MILO_TECNICAS,11,1102,100.0,148


In [5]:
ejemplo = next(f for f in fragmentos if f.source_id == "AESAN_2022" and f.page == 9)
print(f"Documento: {ejemplo.source_id} | página {ejemplo.page} | sección: {ejemplo.section}")
print(f"Identificador: {ejemplo.chunk_id[:16]}…  ({len(ejemplo.text.split())} palabras)\n")
print(ejemplo.text)

Documento: AESAN_2022 | página 9 | sección: Contenido
Identificador: d2c97d07391d5b92…  (142 palabras)

9 HUEVOS CON MODERACIÓN, SEGÚN TUS NECESIDADES Se recomienda consumir un máximo de 4 huevos medianos a la semana. Un huevo mediando M pesa entre 53-63 gramos. ADEMÁS… Acompaña el consumo de huevos con alimentos saludables, como ensaladas, hortalizas, pescados y evita combinaciones con alimentos ricos en grasas saturadas, como algunas carnes procesadas y harinas refinadas, como el pan blanco o los empanados. Los huevos son un alimento asequible y con un impacto medioambiental relativamente bajo. Por razones de bienestar animal, si puedes, consume huevos procedentes de gallinas criadas en libertad (camperas). QUÉ DEBE ESTAR PRESENTE EN UNA DIETA SALUDABLE Y SOSTENIBLE El embarazo, la lactancia y la anorexia asociada al envejecimiento son situaciones con una demanda nutricional elevada El consumo de huevos es muy recomendable en estas etapas de la vida por su alto aporte nutricional y s

In [6]:
receta = next(f for f in fragmentos if f.source_id == "MILO_RECETAS" and f.section.startswith("r023"))
print(f"Documento: {receta.source_id} | sección: {receta.section}\n")
print(receta.text)

Documento: MILO_RECETAS | sección: r023 — Ensalada de pollo y aguacate

## r023 — Ensalada de pollo y aguacate Receta para 1 ración. Ideal para: comida, cena. Tiempo orientativo: 15 minutos. Dificultad: fácil. Dieta: con productos animales o de composición según marca. Ingredientes y cantidades: - Pechuga de pollo sin piel ni hueso, cocinada en líquido: 130 g (fdc_331960). - Lechuga romana verde, cruda: 80 g (fdc_2346389). - Aguacate Hass pelado, crudo: 70 g (fdc_2710824). - Tomate roma, crudo: 100 g (fdc_1999634). - Aceite de oliva para ensalada o cocina (genérico USDA SR): 8 g (fdc_171413). Preparación: 1. Usar pechuga ya cocinada en líquido y pesarla después de cocinar. Si se parte de pollo crudo, cocinarlo hasta 74 °C y pesarlo después. 2. Lavar la lechuga y el tomate. Pelar el aguacate y pesar la parte comestible. 3. Cortar todo en trozos y mezclar con el aceite medido. Alérgenos propios de los ingredientes: ninguno conocido.


## 5. "Así los convierto en vectores": Gemini Embeddings

Un **embedding** es una lista de números que representa el *significado* de un texto. Dos textos que hablan de lo
mismo tienen vectores parecidos aunque usen palabras distintas. Uso `gemini-embedding-001`, que convierte cada
fragmento en un vector de **3072 números**.

Para comprobar que funciona, comparo una pregunta con tres frases usando la **similitud coseno**
(1 = mismo significado, 0 = nada que ver).

In [7]:
import math

embeddings = create_embeddings(settings)
vector = embeddings.embed_query("¿Cuántos huevos puedo comer a la semana?")
print("Dimensiones del vector:", len(vector))
print("Primeros 8 valores:", [round(v, 4) for v in vector[:8]])

def coseno(a, b):
    return sum(x * y for x, y in zip(a, b)) / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

frases = [
    "Se recomienda consumir un máximo de 4 huevos medianos a la semana.",
    "Una tortilla se cuaja a fuego medio hasta alcanzar 71 °C.",
    "El agua del grifo es la bebida de elección.",
]
vectores = embeddings.embed_documents(frases)
pd.DataFrame({"frase": frases, "similitud con la pregunta": [round(coseno(vector, v), 3) for v in vectores]})

Dimensiones del vector: 3072
Primeros 8 valores: [-0.0024, 0.0114, 0.0486, -0.0309, -0.0058, 0.0045, -0.0015, 0.0019]


,frase,similitud con la pregunta
0,Se recomienda consumir un máximo de 4 huevos medianos a la semana.,0.810
1,Una tortilla se cuaja a fuego medio hasta alcanzar 71 °C.,0.584
2,El agua del grifo es la bebida de elección.,0.529


La frase sobre la recomendación semanal de huevos es la más parecida a la pregunta aunque no comparten exactamente
las mismas palabras. Esto es lo que aprovecha la búsqueda semántica.

## 6. "Aquí los guardo": ChromaDB

**ChromaDB** es una base de datos vectorial: guarda cada fragmento junto con su vector y sus metadatos, y permite
buscar los vectores más cercanos a una pregunta. Uso una base **persistente local** en `storage/chroma`
(no necesito un servidor).

- El nombre de la colección depende del modelo de embeddings y de los fragmentos: si cambio un documento, se crea una
  colección nueva y nunca mezclo vectores antiguos.
- `index_chunks` solo genera los embeddings que faltan, así que ejecutar esta celda otra vez no duplica nada.
- Para reconstruir el índice desde cero: `python scripts/build_index.py --rebuild`.

In [8]:
indice = index_chunks(fragmentos, settings)  # Llama a Gemini solo para los fragmentos que aún no están guardados.
guardado = indice.get(include=["metadatas", "documents", "embeddings"])
print("Colección:", indice._collection.name)
print("Vectores guardados en ChromaDB:", len(guardado["ids"]))
print("Dimensión de cada vector:", len(guardado["embeddings"][0]))
print("Fragmentos por documento:", dict(Counter(m["source_id"] for m in guardado["metadatas"])))
print("\nAsí se ve un registro guardado:")
print({"id": guardado["ids"][0][:16] + "…", "metadatos": guardado["metadatas"][0],
       "texto": guardado["documents"][0][:120] + "…", "vector": [round(x, 4) for x in guardado["embeddings"][0][:4]] + ["…"]})

Colección: milo_b51be58f64b32361
Vectores guardados en ChromaDB: 75
Dimensión de cada vector: 3072
Fragmentos por documento: {'AESAN_2022': 35, 'MILO_RECETAS': 29, 'MILO_TECNICAS': 11}

Así se ve un registro guardado:
{'id': '38fe1d4636a67f8f…', 'metadatos': {'version': '2022-12', 'source_sha256': '8a7027defb7395a4b9136487dd9493bb9ff5a43940aafb8ab832dac78b21ccc8', 'source_url': 'https://www.aesan.gob.es/AECOSAN/docs/documentos/nutricion/RECOMENDACIONES_DIETETICAS.pdf', 'section': 'Contenido', 'usage': 'rag_general', 'source_id': 'AESAN_2022', 'page': 3}, 'texto': '3 QUÉ SON LAS RECOMENDACIONES Y PARA QUÉ SIRVEN Las recomendaciones dietéticas sostenibles y de actividad física incluid…', 'vector': [np.float64(0.0117), np.float64(0.0168), np.float64(0.0467), np.float64(-0.0356), '…']}


## 7. "Cuando pregunto algo, estos son los fragmentos que recupera"

Antes de conectar el agente compruebo que la colección responde bien: para cada pregunta, Chroma convierte la
pregunta en un vector con Gemini y devuelve los 4 fragmentos más cercanos (**distancia coseno**: cuanto menor,
más parecido). Verifico automáticamente que el documento esperado aparece entre los resultados.

In [9]:
rag = NutritionRAG(settings)
pruebas = [
    ("¿Cuántas raciones de legumbres debo tomar a la semana?", "AESAN_2022"),
    ("Receta vegetariana con garbanzos", "MILO_RECETAS"),
    ("¿Puedo cambiar el arroz por patata gramo por gramo?", "MILO_TECNICAS"),
]
filas = []
for pregunta, esperado in pruebas:
    resultados = rag.retrieve(pregunta, k=4)
    assert any(r["metadata"]["source_id"] == esperado for r in resultados), pregunta
    for r in resultados:
        filas.append({"pregunta": pregunta, "#": r["citation"], "documento": r["metadata"]["source_id"],
                      "ubicación": location(r["metadata"]), "distancia": round(r["distance"], 3)})
print("✔ Las tres consultas de prueba recuperan su documento esperado")
pd.DataFrame(filas)

✔ Las tres consultas de prueba recuperan su documento esperado


,pregunta,#,documento,ubicación,distancia
0,¿Cuántas raciones de legumbres debo tomar a la semana?,1,AESAN_2022,página 6,0.195
1,¿Cuántas raciones de legumbres debo tomar a la semana?,2,AESAN_2022,página 18,0.245
2,¿Cuántas raciones de legumbres debo tomar a la semana?,3,AESAN_2022,página 4,0.260
3,¿Cuántas raciones de legumbres debo tomar a la semana?,4,AESAN_2022,página 7,0.264
4,Receta vegetariana con garbanzos,1,MILO_RECETAS,r005 — Ensalada de garbanzos,0.234
5,Receta vegetariana con garbanzos,2,MILO_RECETAS,r015 — Ensalada de quinoa con garbanzos,0.252
6,Receta vegetariana con garbanzos,3,MILO_RECETAS,r008 — Lentejas con verduras,0.276
7,Receta vegetariana con garbanzos,4,MILO_RECETAS,r022 — Alubias blancas con espinacas,0.279
8,¿Puedo cambiar el arroz por patata gramo por gramo?,1,MILO_TECNICAS,"Sustituir arroz, pasta, quinoa o patata",0.189
9,¿Puedo cambiar el arroz por patata gramo por gramo?,2,MILO_TECNICAS,"Hervir arroz, pasta y legumbres",0.293


## 8. "Y esta es la respuesta que genera Gemini utilizando esa información"

Los fragmentos recuperados se insertan en el mensaje que recibe Gemini, dentro de etiquetas `<documentos>`, junto con
el **system prompt** de MILO (`src/milo/prompts/system.md`). Gemini debe basarse en ellos y citarlos con `[1]`, `[2]`…
Así se ve exactamente lo que recibe el modelo:

In [10]:
pregunta = "¿Qué recomienda la documentación sobre el consumo de carne y pescado?"
recuperados = rag.retrieve(pregunta, k=4)
print("=== SYSTEM PROMPT (primeras líneas) ===")
print("\n".join(load_system_prompt().splitlines()[:8]), "\n…\n")
print("=== MENSAJE DEL USUARIO QUE RECIBE GEMINI ===")
print(f"<documentos>\n{format_context(recuperados)[:1200]}…\n</documentos>\n\nPregunta: {pregunta}")

=== SYSTEM PROMPT (primeras líneas) ===
# Rol
Eres MILO, un pequeño chef digital experto en cocina casera y educación nutricional general.
Ayudas a decidir qué cocinar con lo que hay en la nevera, a ajustar recetas a un objetivo,
a organizar las comidas de la semana y a entender las recomendaciones de alimentación saludable.

# Personalidad y tono
- Cercano, simpático, práctico y creativo. Puedes abrir con una frase breve y desenfadada cuando la conversación
  es informal, pero inventa una distinta cada vez: nunca repitas la misma muletilla. 
…

=== MENSAJE DEL USUARIO QUE RECIBE GEMINI ===
<documentos>
[1] AESAN_2022 — página 10
10 MENOS CARNE, MEJOR SI ES BLANCA Se recomienda consumir de 0 a un máximo de 3 raciones de carne a la semana , priorizando el consumo de carne blanca de aves y conejo y minimizando el consumo de carne procesada. Cada ración equivale a 100-125 gramos. Ejemplos: 1 filete mediano, ¼ pollo, ¼ conejo. ADEMÁS… Debido a la evidencia científica sobre la relación entr

In [11]:
respuesta = rag.generate(pregunta, recuperados)
from milo.rag import RagAnswer
show_answer(RagAnswer(pregunta, recuperados, respuesta))

PREGUNTA: ¿Qué recomienda la documentación sobre el consumo de carne y pescado?

FRAGMENTOS RECUPERADOS DE CHROMADB:
★[1] AESAN_2022 — página 10 · distancia 0.251
    10 MENOS CARNE, MEJOR SI ES BLANCA Se recomienda consumir de 0 a un máximo de 3 raciones de carne a la semana , priorizando el consumo de carne blanca de aves y conejo y minimizando el consumo de carne procesada. Cada ración equivale a 100-125 gramos. Ejemplos: 1 filete mediano, ¼ pollo, ¼ conejo. A…
 [2] AESAN_2022 — página 3 · distancia 0.263
    3 QUÉ SON LAS RECOMENDACIONES Y PARA QUÉ SIRVEN Las recomendaciones dietéticas sostenibles y de actividad física incluidas en esta Guía permiten cubrir las necesidades nutricionales de la población general promoviendo un estado de salud óptimo. Se han tenido también en cuenta las necesidades nutrici…
★[3] AESAN_2022 — página 8 · distancia 0.264
    8 MÁS PESCADO, MEJOR AZUL Consume al menos 3 raciones a la semana , priorizando el pescado azul (azul: sardinas, boquerones, caball

Las estrellas ★ marcan los fragmentos que Gemini ha citado: puedo comprobar cada afirmación con su fuente.

### El system prompt

El archivo `src/milo/prompts/system.md` define el rol (chef digital de cocina y nutrición general), la personalidad
(cercano y bromista, pero serio con los números), cómo usar los documentos y citarlos, cuándo usar cada herramienta,
cómo tratar la memoria y los límites de seguridad (no diagnostica ni prescribe dietas clínicas). La justificación de
cada decisión está en el README.

In [12]:
print(load_system_prompt())

# Rol
Eres MILO, un pequeño chef digital experto en cocina casera y educación nutricional general.
Ayudas a decidir qué cocinar con lo que hay en la nevera, a ajustar recetas a un objetivo,
a organizar las comidas de la semana y a entender las recomendaciones de alimentación saludable.

# Personalidad y tono
- Cercano, simpático, práctico y creativo. Puedes abrir con una frase breve y desenfadada cuando la conversación
  es informal, pero inventa una distinta cada vez: nunca repitas la misma muletilla.
- Nunca bromeas con los números: cuando das calorías, macros o cantidades eres preciso y sobrio.
- No eres infantil ni pareces un médico.
- Respondes en español por defecto. Si el usuario escribe en catalán o en inglés, respondes en su idioma.
- Respuestas claras y ordenadas: listas cortas, negritas para lo importante, sin relleno.

# Cómo usar los documentos (RAG)
- En cada turno recibes, entre <documentos>, fragmentos recuperados de tu base de conocimiento:
  la guía de AESAN sobre ali

## 9. Agente LangGraph

El RAG por sí solo responde preguntas, pero MILO también tiene que **calcular** (macros, menús, listas). Para eso he
creado un agente con LangGraph. El grafo tiene tres nodos:

```mermaid
flowchart LR
    S([START]) --> R[recuperar<br/>ChromaDB]
    R --> M[milo<br/>Gemini]
    M -- pide herramienta --> H[herramientas<br/>Python]
    H --> M
    M -- respuesta final --> E([END])
```

```
START → recuperar → milo ⇄ herramientas → END
```

1. **recuperar**: en *cada* turno busca en ChromaDB los 4 fragmentos más relevantes. Si la pregunta es muy corta
   ("¿y cuántas calorías tiene?"), le añado la pregunta anterior para que la búsqueda tenga contexto.
2. **milo**: Gemini recibe el system prompt, el perfil del usuario, los fragmentos y el historial. Decide si
   responde directamente o si necesita una herramienta.
3. **herramientas**: ejecuta en Python la herramienta que Gemini ha pedido y le devuelve el resultado. Puede haber
   varias vueltas (por ejemplo, buscar recetas y después ver una).

| Herramienta | Qué hace |
|---|---|
| `buscar_recetas` | Busca recetas compatibles con el perfil; con ingredientes de la nevera dice qué falta |
| `ver_receta` | Ficha completa con macros; cambia raciones o escala a unas kcal objetivo |
| `sustituir_ingrediente` | Cambia un ingrediente y recalcula (por defecto con energía equivalente, nunca gramo a gramo) |
| `calcular_plato` | Calcula un plato libre a partir de alimentos y gramos |
| `plan_semanal` | Menú de 1–7 días ajustado a kcal diarias |
| `lista_compra` | Suma los ingredientes del último menú, agrupados por categorías |
| `guardar_perfil` | Guarda dieta, alergias, alimentos que no gustan, objetivo… en la memoria |
| `buscar_documentos` | Búsqueda extra en ChromaDB si los fragmentos iniciales no bastan |

Las restricciones (dieta, alergias, alimentos que no gustan) **las comprueba Python**, no el modelo.

In [13]:
agente = MiloAgent(rag)
print("Nodos del grafo:", [n for n in agente.graph.get_graph().nodes if not n.startswith("__")])
print("Herramientas:", [t.name for t in agente.tools])
print()
print(agente.graph.get_graph().draw_mermaid())

Nodos del grafo: ['recuperar', 'milo', 'herramientas']
Herramientas: ['buscar_recetas', 'ver_receta', 'sustituir_ingrediente', 'calcular_plato', 'plan_semanal', 'lista_compra', 'guardar_perfil', 'buscar_documentos']

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	recuperar(recuperar)
	milo(milo)
	herramientas(herramientas)
	__end__([<p>__end__</p>]):::last
	__start__ --> recuperar;
	herramientas --> milo;
	milo -.-> __end__;
	milo -. &nbsp;tools&nbsp; .-> herramientas;
	recuperar --> milo;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 10. Memoria conversacional

LangGraph guarda el estado de cada conversación (mensajes, perfil y fragmentos) con un **checkpointer**. Uso
`SqliteSaver`, que lo escribe en `storage/memoria.sqlite`, y un `thread_id` por conversación:

- Dentro de una conversación, MILO recuerda lo que le he dicho ("la receta anterior", "lo que te dije").
- Conversaciones con distinto `thread_id` están separadas (en la app, cada navegador tiene la suya).
- Como es un archivo, la memoria **sobrevive a reiniciar Python**: lo demuestro creando un agente nuevo.

En esta demostración primero cuento mis preferencias, después pido una cena sin repetirlas y por último pido
"la receta anterior" para dos personas sin decir cuál era.

In [14]:
import uuid

# La memoria se guarda en disco: uso identificadores nuevos en cada ejecución para que los ejemplos empiecen de cero.
EJECUCION = uuid.uuid4().hex[:6]
hilo = f"notebook-memoria-{EJECUCION}"

def mostrar_resultados(turno):
    """Resumen de lo que han calculado las herramientas Python en este turno."""
    for r in turno["results"]:
        tipo = r.get("tipo")
        if "error" in r:
            print("  ⚠️ La herramienta devolvió un error:", r["error"])
        elif tipo in {"receta", "plato"}:
            m = r["por_racion"]
            print(f"  🧮 {r['titulo']}: {m['kcal']} kcal · {m['protein_g']} g proteína · {m['carbs_g']} g hidratos · {m['fat_g']} g grasa")
        elif tipo == "sustitucion":
            print(f"  🔁 {r['cambio']['quitar']} → {r['cambio']['poner']} | kcal {r['antes_por_racion']['kcal']} → {r['despues_por_racion']['kcal']}")
        elif tipo == "busqueda":
            print("  🔎", [x["titulo"] for x in r["recetas"]])
        elif tipo == "perfil":
            print("  💾 Perfil guardado:", r["perfil_guardado"])

turno1 = agente.ask("Soy vegetariano y no me gusta el tomate.", thread_id=hilo)
show_turn(turno1); mostrar_resultados(turno1)

Herramientas usadas: guardar_perfil
PREGUNTA: Soy vegetariano y no me gusta el tomate.

FRAGMENTOS RECUPERADOS DE CHROMADB:
 [1] MILO_TECNICAS — Sustituir pollo por pavo, tofu o legumbres · distancia 0.307
    ## Sustituir pollo por pavo, tofu o legumbres Puedo estudiar variantes de un plato de arroz con pavo, tofu o legumbres, pero no prometo equivalencia de proteína…
 [2] MILO_RECETAS — r005 — Ensalada de garbanzos · distancia 0.308
    ## r005 — Ensalada de garbanzos Receta para 1 ración. Ideal para: comida, cena. Tiempo orientativo: 10 minutos. Dificultad: fácil. Dieta: vegana. Ingredientes y…
 [3] MILO_RECETAS — r011 — Arroz con tofu y verduras · distancia 0.311
    ## r011 — Arroz con tofu y verduras Receta para 1 ración. Ideal para: comida, cena. Tiempo orientativo: 25 minutos. Dificultad: media. Dieta: vegana. Ingredient…
 [4] MILO_RECETAS — r013 — Tostada integral con tomate y aceite · distancia 0.313
    ## r013 — Tostada integral con tomate y aceite Receta para 1 ración. Ide

In [15]:
turno2 = agente.ask("Recomiéndame una cena de unas 700 kcal.", thread_id=hilo)
show_turn(turno2); mostrar_resultados(turno2)
print("\nPerfil que recuerda el agente:", turno2["profile"]["restrictions"], turno2["profile"]["disliked_food_ids"])

Herramientas usadas: buscar_recetas, ver_receta
PREGUNTA: Recomiéndame una cena de unas 700 kcal.

FRAGMENTOS RECUPERADOS DE CHROMADB:
 [1] MILO_RECETAS — r007 — Tostadas de aguacate y huevo · distancia 0.264
    ## r007 — Tostadas de aguacate y huevo Receta para 1 ración. Ideal para: desayuno, cena. Tiempo orientativo: 15 minutos. Dificultad: fácil. Dieta: con productos…
 [2] MILO_RECETAS — r017 — Hamburguesa de pavo con boniato y espárragos · distancia 0.275
    ## r017 — Hamburguesa de pavo con boniato y espárragos Receta para 1 ración. Ideal para: comida, cena. Tiempo orientativo: 40 minutos. Dificultad: media. Dieta:…
★[3] MILO_RECETAS — r001 — Bowl de pollo y arroz · distancia 0.276
    ## r001 — Bowl de pollo y arroz Receta para 1 ración. Ideal para: comida, cena. Tiempo orientativo: 20 minutos. Dificultad: fácil. Dieta: con productos animales…
 [4] MILO_RECETAS — r025 — Salteado de tofu con brócoli y arroz integral · distancia 0.278
    ## r025 — Salteado de tofu con brócoli y 

In [16]:
turno3 = agente.ask("Hazme la receta anterior para 2 personas.", thread_id=hilo)
show_turn(turno3); mostrar_resultados(turno3)

Herramientas usadas: ver_receta
PREGUNTA: Hazme la receta anterior para 2 personas.

FRAGMENTOS RECUPERADOS DE CHROMADB:
 [1] MILO_RECETAS — r002 — Tortilla de espinacas · distancia 0.266
    ## r002 — Tortilla de espinacas Receta para 1 ración. Ideal para: cena. Tiempo orientativo: 15 minutos. Dificultad: fácil. Dieta: vegetariana. Ingredientes y ca…
 [2] MILO_RECETAS — r025 — Salteado de tofu con brócoli y arroz integral · distancia 0.272
    ## r025 — Salteado de tofu con brócoli y arroz integral Receta para 1 ración. Ideal para: comida, cena. Tiempo orientativo: 35 minutos. Dificultad: media. Dieta…
 [3] MILO_RECETAS — r001 — Bowl de pollo y arroz · distancia 0.277
    ## r001 — Bowl de pollo y arroz Receta para 1 ración. Ideal para: comida, cena. Tiempo orientativo: 20 minutos. Dificultad: fácil. Dieta: con productos animales…
★[4] MILO_RECETAS — r011 — Arroz con tofu y verduras · distancia 0.278
    ## r011 — Arroz con tofu y verduras Receta para 1 ración. Ideal para: comida, cen

In [17]:
# Simulo un reinicio: creo un agente NUEVO que lee la misma base storage/memoria.sqlite.
agente_reiniciado = MiloAgent(rag)
turno4 = agente_reiniciado.ask("¿Qué te he dicho antes sobre mi dieta y lo que no me gusta?", thread_id=hilo)
print(turno4["answer"])
print("\nMensajes guardados en este hilo:", len(turno4["messages"]))

# Otra conversación no ve nada de lo anterior:
otro = agente.ask("¿Sabes si soy vegetariano?", thread_id=f"otro-{uuid.uuid4().hex[:6]}")
print("\nOtra conversación →", otro["answer"][:300])

¡Claro que me acuerdo! Me contaste que eres **vegetariano** y que no te gusta el **tomate**.

Mensajes guardados en este hilo: 16



Otra conversación → ¡Hola! Echando un vistazo a tus datos, **no tengo registrado** si sigues una dieta vegetariana o no. 

Si quieres, puedes decírmelo y lo guardo para que todas las recetas y menús que te proponga se adapten a ti.


## 11. Ejemplos documentados

Seis consultas que cubren las funciones principales. Para cada una muestro las herramientas usadas, los fragmentos
recuperados (★ = citados) y la respuesta de MILO.

| # | Pregunta | Qué demuestra |
|---|---|---|
| 1 | ¿Qué recomienda la documentación sobre una alimentación equilibrada? | RAG puro sobre la guía AESAN |
| 2 | Tengo pollo, arroz, aguacate y huevos… cena de 800 kcal con proteína | Modo nevera + escalado con macros |
| 3 | No tengo aguacate, ¿por qué lo sustituyo en las tostadas? | Sustitución y recálculo |
| 4 | Menú de 3 días de 2.500 kcal + lista de la compra | Planificación y memoria entre turnos |
| 5 | ¿Yogur sin lactosa con alergia a la leche? | Seguridad, documento de técnicas |
| 6 | Quants ous puc menjar a la setmana? | Idioma del usuario (catalán) |

In [18]:
ejemplos = agente.ask("¿Qué recomienda la documentación sobre una alimentación equilibrada?", thread_id=f"ej-1-{EJECUCION}")
show_turn(ejemplos)

Herramientas usadas: ninguna (respuesta solo con RAG)
PREGUNTA: ¿Qué recomienda la documentación sobre una alimentación equilibrada?

FRAGMENTOS RECUPERADOS DE CHROMADB:
★[1] AESAN_2022 — página 3 · distancia 0.243
    3 QUÉ SON LAS RECOMENDACIONES Y PARA QUÉ SIRVEN Las recomendaciones dietéticas sostenibles y de actividad física incluidas en esta Guía permiten cubrir las nece…
★[2] AESAN_2022 — página 18 · distancia 0.263
    AGUA El agua es la bebida de elección en una dieta saludable. Bebe agua siempre que tengas sed. Las necesidades pueden aumentar en situaciones de altas temperat…
 [3] AESAN_2022 — página 7 · distancia 0.264
    QUÉ DEBE ESTAR PRESENTE EN UNA DIETA SALUDABLE Y SOSTENIBLE
★[4] AESAN_2022 — página 4 · distancia 0.276
    4 QUÉ DEBE ESTAR PRESENTE EN UNA DIETA SALUDABLE Y SOSTENIBLE MÁS HORTALIZAS Y FRUTAS Consume un mínimo de 5 raciones al día (de las cuales, al menos 3 de horta…

RESPUESTA DE MILO:
¡Hola! Qué buena pregunta para empezar. 

Basándonos en la guía de 

In [19]:
t = agente.ask("Tengo pollo, arroz, aguacate y huevos. Quiero preparar una cena de unas 800 kcal y bastante proteína.",
               thread_id=f"ej-2-{EJECUCION}")
show_turn(t); mostrar_resultados(t)

Herramientas usadas: buscar_recetas, ver_receta
PREGUNTA: Tengo pollo, arroz, aguacate y huevos. Quiero preparar una cena de unas 800 kcal y bastante proteína.

FRAGMENTOS RECUPERADOS DE CHROMADB:
 [1] MILO_RECETAS — r023 — Ensalada de pollo y aguacate · distancia 0.228
    ## r023 — Ensalada de pollo y aguacate Receta para 1 ración. Ideal para: comida, cena. Tiempo orientativo: 15 minutos. Dificultad: fácil. Dieta: con productos a…
★[2] MILO_RECETAS — r001 — Bowl de pollo y arroz · distancia 0.229
    ## r001 — Bowl de pollo y arroz Receta para 1 ración. Ideal para: comida, cena. Tiempo orientativo: 20 minutos. Dificultad: fácil. Dieta: con productos animales…
 [3] MILO_RECETAS — r007 — Tostadas de aguacate y huevo · distancia 0.237
    ## r007 — Tostadas de aguacate y huevo Receta para 1 ración. Ideal para: desayuno, cena. Tiempo orientativo: 15 minutos. Dificultad: fácil. Dieta: con productos…
 [4] MILO_RECETAS — r024 — Huevos con patata y espárragos · distancia 0.271
    ## r024 — 

In [20]:
t = agente.ask("No tengo aguacate, ¿por qué lo puedo sustituir en las tostadas de aguacate y huevo?", thread_id=f"ej-3-{EJECUCION}")
show_turn(t); mostrar_resultados(t)

Herramientas usadas: sustituir_ingrediente
PREGUNTA: No tengo aguacate, ¿por qué lo puedo sustituir en las tostadas de aguacate y huevo?

FRAGMENTOS RECUPERADOS DE CHROMADB:
 [1] MILO_RECETAS — r007 — Tostadas de aguacate y huevo · distancia 0.262
    ## r007 — Tostadas de aguacate y huevo Receta para 1 ración. Ideal para: desayuno, cena. Tiempo orientativo: 15 minutos. Dificultad: fácil. Dieta: con productos…
 [2] MILO_TECNICAS — Saltear verduras y preparar un revuelto · distancia 0.318
    ## Saltear verduras y preparar un revuelto En un salteado corto los ingredientes de forma uniforme y tengo en cuenta que no tardan lo mismo en cocinarse. Puedo …
★[3] MILO_TECNICAS — Sustituir arroz, pasta, quinoa o patata · distancia 0.327
    ## Sustituir arroz, pasta, quinoa o patata Estos alimentos pueden ocupar un papel parecido como acompañamiento, pero difieren en humedad, preparación y composic…
 [4] MILO_TECNICAS — Sustituir pollo por pavo, tofu o legumbres · distancia 0.327
    ## Sustitu

In [21]:
t = agente.ask("Hazme un menú de 3 días de unas 2500 kcal al día.", thread_id=f"ej-4-{EJECUCION}")
show_turn(t)
plan = next(r for r in t["results"] if r.get("tipo") == "plan")
pd.DataFrame([{"día": d["dia"], "comida": m["comida"], "receta": m["titulo"], "kcal": m["kcal"], "proteína (g)": m["protein_g"]}
              for d in plan["dias"] for m in d["comidas"]])

Herramientas usadas: plan_semanal
PREGUNTA: Hazme un menú de 3 días de unas 2500 kcal al día.

FRAGMENTOS RECUPERADOS DE CHROMADB:
 [1] MILO_RECETAS — r025 — Salteado de tofu con brócoli y arroz integral · distancia 0.288
    ## r025 — Salteado de tofu con brócoli y arroz integral Receta para 1 ración. Ideal para: comida, cena. Tiempo orientativo: 35 minutos. Dificultad: media. Dieta…
 [2] MILO_RECETAS — r024 — Huevos con patata y espárragos · distancia 0.304
    ## r024 — Huevos con patata y espárragos Receta para 1 ración. Ideal para: cena. Tiempo orientativo: 30 minutos. Dificultad: fácil. Dieta: vegetariana. Ingredie…
 [3] MILO_RECETAS — r027 — Crema de calabacín · distancia 0.304
    ## r027 — Crema de calabacín Receta para 1 ración. Ideal para: cena. Tiempo orientativo: 25 minutos. Dificultad: fácil. Dieta: vegana. Ingredientes y cantidades…
 [4] MILO_RECETAS — r011 — Arroz con tofu y verduras · distancia 0.306
    ## r011 — Arroz con tofu y verduras Receta para 1 ración. Ideal p

,día,comida,receta,kcal,proteína (g)
0,Lunes,Desayuno,Avena con yogur y plátano,754.0,25.4
1,Lunes,Comida,Bowl de pollo y arroz,997.0,72.9
2,Lunes,Cena,Tortilla de espinacas,659.0,42.9
3,Martes,Desayuno,Tostadas de aguacate y huevo,744.0,36.0
4,Martes,Comida,Pasta con atún y tomate,996.0,68.8
5,Martes,Cena,Ensalada de garbanzos,730.0,28.4
6,Miércoles,Desayuno,Yogur con frutos rojos y nueces,740.0,24.6
7,Miércoles,Comida,Salmón al horno con patata,997.0,66.1
8,Miércoles,Cena,Wrap de pavo,749.0,59.7


In [22]:
t = agente.ask("Genera la lista de la compra de ese menú.", thread_id=f"ej-4-{EJECUCION}")  # Mismo hilo: recuerda el menú.
print("Herramientas usadas:", t["tools"])
lista = next(r for r in t["results"] if r.get("tipo") == "lista_compra")
pd.DataFrame([{"categoría": c["categoria"], "producto": i["nombre"], "gramos": i["gramos"]}
              for c in lista["categorias"] for i in c["items"]])

Herramientas usadas: ['lista_compra']


,categoría,producto,gramos
0,"Carne, pescado y marisco","Atún claro al natural en conserva, escurrido",215.0
1,"Carne, pescado y marisco","Pavo picado 93% magro, cocinado en sartén",185.0
2,"Carne, pescado y marisco","Pechuga de pollo sin piel ni hueso, cruda",275.0
3,"Carne, pescado y marisco","Salmón atlántico de cultivo, crudo",255.0
4,Huevos y lácteos,"Huevo entero grande, crudo, sin cáscara",455.0
5,Huevos y lácteos,Yogur natural de leche entera,685.0
6,"Cereales, pan y pasta","Arroz blanco de grano largo sin enriquecer, seco",135.0
7,"Cereales, pan y pasta","Copos de avena integral tradicionales, secos",100.0
8,"Cereales, pan y pasta","Espaguetis enriquecidos, secos",160.0
9,"Cereales, pan y pasta",Pan comercial 100% integral de trigo,115.0


In [23]:
t = agente.ask("¿Puedo tomar un yogur sin lactosa si tengo alergia a la leche?", thread_id=f"ej-5-{EJECUCION}")
show_turn(t)

Herramientas usadas: ninguna (respuesta solo con RAG)
PREGUNTA: ¿Puedo tomar un yogur sin lactosa si tengo alergia a la leche?

FRAGMENTOS RECUPERADOS DE CHROMADB:
★[1] MILO_TECNICAS — Sustituir yogures y lácteos · distancia 0.230
    ## Sustituir yogures y lácteos Un yogur vegetal puede modificar textura, sabor y composición. No afirmo que tenga la misma proteína que uno lácteo. Compruebo si…
★[2] MILO_TECNICAS — Referencias y alcance · distancia 0.307
    ## Referencias y alcance [1] FoodSafety.gov, temperaturas internas mínimas: https://www.foodsafety.gov/food-safety-charts/safe-minimum-internal-temperatures [2]…
 [3] AESAN_2022 — página 10 · distancia 0.331
    QUÉ DEBE ESTAR PRESENTE EN UNA DIETA SALUDABLE Y SOSTENIBLE LECHE Y PRODUCTOS LÁCTEOS CON MODERACIÓN, SEGÚN TUS NECESIDADES Se recomienda consumir un máximo de …
 [4] MILO_RECETAS — r012 — Yogur con frutos rojos y nueces · distancia 0.356
    ## r012 — Yogur con frutos rojos y nueces Receta para 1 ración. Ideal para: desayun

In [24]:
t = agente.ask("Quants ous puc menjar a la setmana?", thread_id=f"ej-6-{EJECUCION}")
show_turn(t)

Herramientas usadas: ninguna (respuesta solo con RAG)
PREGUNTA: Quants ous puc menjar a la setmana?

FRAGMENTOS RECUPERADOS DE CHROMADB:
★[1] AESAN_2022 — página 9 · distancia 0.231
    9 HUEVOS CON MODERACIÓN, SEGÚN TUS NECESIDADES Se recomienda consumir un máximo de 4 huevos medianos a la semana. Un huevo mediando M pesa entre 53-63 gramos. A…
 [2] AESAN_2022 — página 18 · distancia 0.313
    AGUA El agua es la bebida de elección en una dieta saludable. Bebe agua siempre que tengas sed. Las necesidades pueden aumentar en situaciones de altas temperat…
 [3] AESAN_2022 — página 10 · distancia 0.316
    10 MENOS CARNE, MEJOR SI ES BLANCA Se recomienda consumir de 0 a un máximo de 3 raciones de carne a la semana , priorizando el consumo de carne blanca de aves y…
 [4] AESAN_2022 — página 10 · distancia 0.325
    QUÉ DEBE ESTAR PRESENTE EN UNA DIETA SALUDABLE Y SOSTENIBLE LECHE Y PRODUCTOS LÁCTEOS CON MODERACIÓN, SEGÚN TUS NECESIDADES Se recomienda consumir un máximo de …

RESPUESTA DE MI

### Celda interactiva

Escribo una pregunta en la variable y ejecuto la celda. Mantengo el mismo `thread_id` para que MILO recuerde la
conversación. (La interfaz cómoda está en la app de Streamlit: `streamlit run app.py`.)

In [25]:
mi_pregunta = "¿Qué desayuno rápido me recomiendas con mucha proteína?"
turno = agente.ask(mi_pregunta, thread_id="celda-interactiva")
show_turn(turno); mostrar_resultados(turno)

Herramientas usadas: buscar_recetas
PREGUNTA: ¿Qué desayuno rápido me recomiendas con mucha proteína?

FRAGMENTOS RECUPERADOS DE CHROMADB:
 [1] MILO_RECETAS — r003 — Avena con yogur y plátano · distancia 0.290
    ## r003 — Avena con yogur y plátano Receta para 1 ración. Ideal para: desayuno. Tiempo orientativo: 5 minutos. Dificultad: fácil. Dieta: vegetariana. Ingredient…
 [2] MILO_RECETAS — r018 — Batido de plátano, avena y leche · distancia 0.299
    ## r018 — Batido de plátano, avena y leche Receta para 1 ración. Ideal para: desayuno. Tiempo orientativo: 5 minutos. Dificultad: fácil. Dieta: vegetariana. Ing…
 [3] MILO_RECETAS — r007 — Tostadas de aguacate y huevo · distancia 0.301
    ## r007 — Tostadas de aguacate y huevo Receta para 1 ración. Ideal para: desayuno, cena. Tiempo orientativo: 15 minutos. Dificultad: fácil. Dieta: con productos…
★[4] MILO_RECETAS — r020 — Yogur griego con arándanos y pipas · distancia 0.309
    ## r020 — Yogur griego con arándanos y pipas Receta para

## 12. Conclusiones

**Qué he construido.** Un asistente experto en cocina y nutrición que demuestra todo el ciclo RAG con datos reales:
tres documentos → 75 fragmentos → embeddings de Gemini → ChromaDB → recuperación con fuentes → respuesta de Gemini
citada. Encima, un agente de LangGraph con ocho herramientas Python y memoria persistente por conversación, y una
app de Streamlit con chat, "Mi nevera", plan semanal y perfil.

**Qué he aprendido.**
- El chunking importa: quitar el índice del PDF mejoró claramente los resultados de la búsqueda.
- Separar responsabilidades da fiabilidad: el RAG aporta conocimiento y Python hace las cuentas; así las calorías
  nunca dependen de lo que "recuerde" el modelo.
- LangGraph hace visible el flujo del agente: cada nodo es una función que puedo probar por separado.
- El system prompt necesita iteraciones: tuve que aclarar que los datos de las herramientas no se citan con [n] y
  quitar una frase de ejemplo del tono, porque Gemini la copiaba literalmente en casi todas las respuestas.

**Limitaciones.**
- El catálogo tiene 70 alimentos y el recetario 28 recetas: no cubre cualquier ingrediente.
- Los macros son estimaciones con alimentos de referencia USDA, sin pérdidas de cocción ni marcas concretas.
- La fibra aparece como "sin dato" cuando algún ingrediente no la publica en USDA (no la invento como cero).
- El plan semanal escala recetas con un factor uniforme; no es un optimizador nutricional ni una dieta clínica.
- Depende de la disponibilidad de la API de Gemini (por eso configuré reintentos y un modelo de respaldo).

**Mejoras posibles.** Ampliar catálogo y recetario, añadir búsqueda híbrida (palabras clave + vectores), evaluar el
RAG con un conjunto de preguntas de referencia y permitir que el usuario guarde sus propias recetas.